### Rag Pipelines - Data ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\Anchal Pathania\Documents\ProjectRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the data/pdf_files directory

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDFs in a directory"""

    all_documents = [] ### a bucket to hold all the documents 

    pdf_dir = Path(pdf_directory) ## Convert to Path object for easier handling

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf")) ## This will find all pdf files in the directory and subdirectories

    print(f"Found {len(pdf_files)} PDF files to process") ## Print the number of PDF files found

    for pdf_file in pdf_files: ## Loop through each PDF file
        print(f"\nProcessing: {pdf_file.name}")## Print the name of the file being processed

        try:
            loader = PyPDFLoader(str(pdf_file)) ## Create a loader for the PDF file
            documents = loader.load()## Load the PDF and split into documents (pages)

            # Add source information to metadata
            for doc in documents:   ### Loop through each document (page) and add metadata
                doc.metadata["source_file"] = pdf_file.name  ### Add source file name to metadata
                doc.metadata["file_type"] = "pdf"  ### Add file type to metadata

            all_documents.extend(documents)  ## Add the documents from this PDF to the main list
            print(f" ✓ Loaded {len(documents)} pages") ## Print the number of pages loaded from this PDF

        except Exception as e: ## Catch any exceptions that occur during loading and print an error message
            print(f" ✗ Error: {e}") ### Print the error message

    print(f"\nTotal documents loaded: {len(all_documents)}") ### Print the total number of documents loaded from all PDFs
    return all_documents ### Return the list of all documents


all_pdf_documents = process_all_pdfs("../data/Pdfs")


Found 5 PDF files to process

Processing: content_tag prediction.pdf
 ✓ Loaded 25 pages

Processing: Cover Letter_support the ServiceNow platform team.pdf
 ✓ Loaded 1 pages

Processing: CoverLetter_Digital Maintenance with a focus on Artificial Intelligence.pdf
 ✓ Loaded 1 pages

Processing: Experience Letter -Anchal Pathania- 01.pdf
 ✓ Loaded 1 pages

Processing: Gold Prices.pdf
 ✓ Loaded 52 pages

Total documents loaded: 80


In [3]:

all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Services', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-08-28T06:23:34+00:00', 'author': 'ANCHAL PATHANIA', 'moddate': '2024-08-28T06:23:37+00:00', 'title': 'NLP BASED TAG PREDICTION', 'source': '..\\data\\Pdfs\\content_tag prediction.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': 'content_tag prediction.pdf', 'file_type': 'pdf'}, page_content='Contents \n1. ABSTRACT ....................................................................................................................................... 2 \n2. INTRODUCTION ............................................................................................................................. 2 \n2.1 What is Stack Overflow? .............................................................................................................. 2 \n2.2 What are tags? ........................................................................................................

In [4]:
### Text splitting into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 80 documents into 111 chunks

Example chunk:
Content: Contents 
1. ABSTRACT ....................................................................................................................................... 2 
2. INTRODUCTION ..........................
Metadata: {'producer': 'Adobe PDF Services', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-08-28T06:23:34+00:00', 'author': 'ANCHAL PATHANIA', 'moddate': '2024-08-28T06:23:37+00:00', 'title': 'NLP BASED TAG PREDICTION', 'source': '..\\data\\Pdfs\\content_tag prediction.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': 'content_tag prediction.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Services', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-08-28T06:23:34+00:00', 'author': 'ANCHAL PATHANIA', 'moddate': '2024-08-28T06:23:37+00:00', 'title': 'NLP BASED TAG PREDICTION', 'source': '..\\data\\Pdfs\\content_tag prediction.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': 'content_tag prediction.pdf', 'file_type': 'pdf'}, page_content='Contents \n1. ABSTRACT ....................................................................................................................................... 2 \n2. INTRODUCTION ............................................................................................................................. 2 \n2.1 What is Stack Overflow? .............................................................................................................. 2 \n2.2 What are tags? ........................................................................................................

### Embedding and vectorstore DB

In [6]:
import numpy as np ### For numerical operations 
from sentence_transformers import SentenceTransformer ### For generating embeddings
import chromadb ## For vector database management
from chromadb.config import Settings ### For configuring ChromaDB
import uuid ### For generating unique IDs
from typing import List, Dict, Any, Tuple ### For type annotations
from sklearn.metrics.pairwise import cosine_similarity ### For calculating cosine similarity

In [7]:
### here we will create a class to manage the embedding generation using SentenceTransformer.
### This class will load the model and provide a method to generate embeddings for a list of texts.
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3737.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


C:\Users\Anchal Pathania\AppData\Local\Temp\ipykernel_19628\3865728224.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:

"""Now we will create a class to manage the vector store using ChromaDB. 
This class will handle the initialization of the vector store, adding documents and their embeddings, and retrieving similar
documents based on a query embedding."""
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): ### Loop through each document and its corresponding embedding
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" ### Create a unique ID using uuid and the index
            ids.append(doc_id)### Add the generated ID to the list of IDs
            
            # Prepare metadata
            metadata = dict(doc.metadata) ### Start with the existing metadata from the document
            metadata['doc_index'] = i### Add the document index to metadata for reference
            metadata['content_length'] = len(doc.page_content)### Add the length of the content to metadata
            metadatas.append(metadata) ### Add the prepared metadata to the list of metadatas
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 333


In [9]:
chunks

[Document(metadata={'producer': 'Adobe PDF Services', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-08-28T06:23:34+00:00', 'author': 'ANCHAL PATHANIA', 'moddate': '2024-08-28T06:23:37+00:00', 'title': 'NLP BASED TAG PREDICTION', 'source': '..\\data\\Pdfs\\content_tag prediction.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': 'content_tag prediction.pdf', 'file_type': 'pdf'}, page_content='Contents \n1. ABSTRACT ....................................................................................................................................... 2 \n2. INTRODUCTION ............................................................................................................................. 2 \n2.1 What is Stack Overflow? .............................................................................................................. 2 \n2.2 What are tags? ........................................................................................................

In [10]:
### Convert the text to embeddings 
texts = [doc.page_content for doc in chunks] ### Extract the text content from each chunk to create a list of texts
embeddings = embedding_manager.generate_embeddings(texts) ### Generate embeddings for the list of texts
vectorstore.add_documents(chunks, embeddings) ### Add the chunks and their embeddings to the vector store

Generating embeddings for 111 texts...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.51it/s]


Generated embeddings with shape: (111, 384)
Adding 111 documents to vector store...
Successfully added 111 documents to vector store
Total documents in collection: 444


''' Now we have to work on the retrieval part. We will create a method in the VectorStore class to retrieve 
similar documents based on a query embedding. This method will take a query string, generate its embedding, and then 
 use cosine similarity to find the most similar documents in the vector store.
Retrieval is the step where the system finds the most relevant information from stored data to answer a user’s question.

The user query is converted into numbers (embedding)
It is compared with stored document embeddings
The most similar pieces (context) are selected
This context is sent to the AI (LLM) to generate a better answer

👉 In short: find relevant data → give it to AI → get accurate answer '''


### Retriever Pipeline From VectorStore 

In [11]:
from typing import List, Dict, Any

class RAGRetriever:
    """
    This class is responsible for:
    👉 Taking a user query
    👉 Finding the most relevant documents from the vector database
    """

    def __init__(self, vector_store, embedding_manager):
        """
        Setup the retriever

        vector_store:
            - This is where all document embeddings are stored (vector DB)

        embedding_manager:
            - This converts text → numerical vectors (embeddings)
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Main function to get relevant documents

        query:
            - The user question (e.g., "What is cloud computing?")

        top_k:
            - How many top results to return (default = 5)

        score_threshold:
            - Minimum similarity required to keep a result
        """

        print(f"\n🔍 Searching for: {query}")
        print(f"Top results requested: {top_k}")

        try:
            # ---------------------------------------------------
            # STEP 1: Convert user query into embedding (numbers)
            # ---------------------------------------------------
            # Why? → So we can compare it with stored document vectors
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]

            # ---------------------------------------------------
            # STEP 2: Search vector database
            # ---------------------------------------------------
            # It finds documents whose embeddings are closest to the query
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],  # convert numpy → list
                n_results=top_k
            )

            # ---------------------------------------------------
            # STEP 3: Prepare empty list to store final results
            # ---------------------------------------------------
            retrieved_docs = []

            # Check if any documents were returned
            if results.get('documents') and results['documents'][0]:

                # Extract different parts of results
                documents = results['documents'][0]      # actual text
                metadatas = results.get('metadatas', [[]])[0]  # extra info
                distances = results.get('distances', [[]])[0]  # similarity measure
                ids = results.get('ids', [[]])[0]        # document IDs

                # ---------------------------------------------------
                # STEP 4: Loop through results
                # ---------------------------------------------------
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance → similarity score
                    # (lower distance = more similar)
                    similarity_score = 1 - distance

                    # Keep only results above threshold
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,                     # unique document ID
                            "content": document,              # actual text content
                            "metadata": metadata,             # extra info (optional)
                            "similarity_score": similarity_score,  # how relevant
                            "distance": distance,             # raw distance
                            "rank": i + 1                     # position in results
                        })

                print(f" Retrieved {len(retrieved_docs)} relevant documents")

            else:
                print("No documents found")

            # ---------------------------------------------------
            # STEP 5: Return final results
            # ---------------------------------------------------
            return retrieved_docs

        except Exception as e:
            # If something goes wrong, don't crash → show error
            print(f" Error during retrieval: {e}")
            return []


# ---------------------------------------------------
# HOW TO USE THIS CLASS
# ---------------------------------------------------

# Make sure these are already created:
# vectorstore = your vector database
# embedding_manager = your embedding generator

# Create retriever object
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


 

In [12]:
## here we are just testing the retriever with a sample query. You can replace the query with any question you want
## to ask based on the documents you have loaded into the vector store.
results = rag_retriever.retrieve("What is stack overflow?")
print(results)


🔍 Searching for: What is stack overflow?
Top results requested: 5
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 21.62it/s]

Generated embeddings with shape: (1, 384)
 Retrieved 5 relevant documents
[{'id': 'doc_ecbd73f9_9', 'content': 'Stack Exchange, which is a network of question -and-answer websites on several topics in \ndiverse fields , each site covering a specific topic where questions, answers , and users are \nsubject to a reputation award process. Stack Overflow features questions and answers on a wide \nrange of topics under computer programming. The website serves as a platform for users to ask \nand answer questions, and through membership and active participation they can earn \nreputation points and bad ges. By increasing these points and badges, users can unlock new \nprivileges like the ability to vote questions and answers up or down, and edit and comment on \nother people’s posts. \n2.2 What are tags? \n Tags are simple keywords or labels that are assigned to a piece of information that helps \nto describe an item and allows it to be found again while browsing or searching.  The tags in \

### Integration Vectordb Context pipeline with LLM output

In [13]:
from langchain_groq import ChatGroq ###API key = your permission to talk to the AI
from dotenv import load_dotenv
import os

In [14]:
import sys
print(sys.executable)

c:\Users\Anchal Pathania\Documents\ProjectRAG\.venv\Scripts\python.exe


In [15]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content 

In [16]:
answer=rag_simple("what is gold prices?", rag_retriever,llm)
print(answer)


🔍 Searching for: what is gold prices?
Top results requested: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.59it/s]

Generated embeddings with shape: (1, 384)
 Retrieved 3 relevant documents


Gold prices refer to the cost or value of gold, which can be influenced by various factors such as supply and demand, interest rates, and domestic demand, particularly during festivals and wedding seasons.


### Enhanced RAG Pipeline Features 

In [18]:
'''Why we need this enhanced RAG pipeline

At its core, this code exists to solve a simple but critical problem:

👉 “How do we make an AI answer based on real data—reliably, transparently, and safely?”

A basic RAG system can answer questions, but it has serious gaps:

It may use weak or irrelevant documents
You don’t know where the answer came from
You can’t tell if the answer is trustworthy
Debugging is painful

This enhanced version fixes those issues.'''



# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what are gold prices?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


🔍 Searching for: what are gold prices?
Top results requested: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.51it/s]

Generated embeddings with shape: (1, 384)
 Retrieved 3 relevant documents


Answer: Gold prices refer to the cost or value of gold, which can be influenced by various factors such as supply and demand, interest rates, and domestic demand, particularly during festivals and wedding seasons.
Sources: [{'source': 'Gold Prices.pdf', 'page': 10, 'score': 0.2835269570350647, 'preview': '• This is because the flow of cash in the market is increased \nwhile the supply of gold goes down.  \n  \n \n6.4 Supply & Demand  \n  It costs more to get less gold.  \n  Thus, the increase in cost of gold production sometimes \nresults in higher gold prices.  \n \n \n6.5 Interest Rates  \n• When the interest...'}, {'source': 'Gold Prices.pdf', 'page': 10, 'score': 0.2835269570350647, 'preview': '• This is because the flow of cash in the market is increased \nwhile the supply of gold goes down.  \n  \n \n6.4 Supply & Demand  \n  It costs more to get less gold.  \n  Thus, the increase in cost of gold production sometimes \nresults in higher gold prices.  \n \n \n6.5 Interest Rates  \n

In [20]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what are gold prices", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


🔍 Searching for: what are gold prices
Top results requested: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.11it/s]

Generated embeddings with shape: (1, 384)
 Retrieved 3 relevant documents
Streaming answer:
Use the following context to answer the question concisely.
Context:
• This is because the flow of cash in the market is increased 
while the supply of gold goes down.  
  
 
6.4 Supply & Demand  
  It costs more to get less gold.  
  Thus, the increase in cost of gold production sometimes 
results in higher gold price

s.  
 
 
6.5 Interest Rates  
• When the interest rates of banks are lowered, more people 
borrow more loans which leads to more money in the economy.  
• Thus, consumption increases which leads to inflation and that 
eventually results in the rise in gold price.  
 
 
6.6 Jewellery& Other domestic demand  
• The price of gold increases during the different festivals and 
wedding seasons since their demand increases.  
• Demand for gold also comes from other domestic 
manufacturing needs.

• This is because the flow of cash in the market is increased 
while the supply of gold goes down.  
  
 
6.4 Supply & Demand  
  It costs more to get less gold.  
  Thus, the increase in cost of gold production sometimes 
results in higher gold prices.  
 
 
6.5 Interest Rates  
• When the interest rates of banks are lowered, more people 
borrow more loans which leads to more money in the economy.  
• Thus, consumption increases which leads to inflation and that 
eventually results in the rise in go